# Problem Statement
#### About Company
Dream Housing Finance company deals in all home loans. They have presence across all urban, semi urban and rural areas. Customer first apply for home loan after that company validates the customer eligibility for loan.

#### Problem
Company wants to automate the loan eligibility process (real time) based on customer detail provided while filling online application form. These details are Gender, Marital Status, Education, Number of Dependents, Income, Loan Amount, Credit History and others. To automate this process, they have given a problem to identify the customers segments, those are eligible for loan amount so that they can specifically target these customers. 

Note: 

Evaluation Metric is accuracy i.e. percentage of loan approval you correctly predict.

In [ ]:
# Importing required libraries
import pandas
import numpy
import matplotlib


import sklearn
from sklearn import model_selection # for splitting into train and test
from sklearn import linear_model # for logistic model
from sklearn import discriminant_analysis # for LinearDiscriminantAnalysis model
from sklearn import tree # for decision tree
from sklearn import naive_bayes
import warnings

# Ignore warnings
warnings.filterwarnings("ignore")

In [ ]:
dataset=pandas.read_csv('/kaggle/input/loan-prediction/Loan_Train.txt')

# Summarize the Dataset

In this step we are going to take a look at the data a few different ways:

1. Dimensions of the dataset.
2. Look at the data itself.
3. Statistical summary of all attributes.
4. Breakdown of the data by the class variable.
5. Non

In [ ]:
# 1. Dimensions of Dataset to see rows and cols count
print(dataset.shape)

In [ ]:
# 2. Look at the data itself.
display(dataset.head(10))

In [ ]:
# 3. Statistical Summary
display(dataset.describe()) 
# it will identify numeric data automatically
# Count is excluded with NAs

In [ ]:
# 4. Class Distribution
print(dataset.groupby('Loan_Status').size())

i.e. 31% Loan_status NO and 68% loan status YES

In [ ]:
# we call also check the distribution by following way
print(dataset['Gender'].value_counts(), end='\n\n')
print(dataset['Married'].value_counts(), end='\n\n')
print(dataset['Education'].value_counts(), end='\n\n')
print(dataset['Self_Employed'].value_counts(), end='\n\n')
print(dataset['Property_Area'].value_counts(), end='\n\n')

# Data Visualization

We are going to look at two types of plots:

1. Univariate plots to better understand each attribute.
2. Multivariate plots to better understand the relationships between attributes.

 ### Univariate Plots

In [ ]:
# box and whisker plots
dataset.plot(kind='box', subplots=True, layout=(2,4), sharex=False, sharey=False)
matplotlib.pyplot.show()

#layout=(row,col)

In [ ]:
# histograms
dataset.hist()
matplotlib.pyplot.show()

In [ ]:
dataset.boxplot(column='ApplicantIncome', by = 'Education')
matplotlib.pyplot.show()

In [ ]:
dataset['LoanAmount'].hist(bins=30) # 30 bins

In [ ]:
# playing with categorical variable

In [ ]:
temp1 = dataset['Credit_History'].value_counts(ascending=True)
temp2 = dataset.pivot_table(values='Loan_Status',index=['Credit_History'],aggfunc=lambda x: x.map({'Y':1,'N':0}).mean())

print ('Frequency Table for Credit History:') 
display (temp1)

print ('\nProbility of getting loan for each Credit History class:')
display (temp2)

# Data Munging
Data Manupulating
i.e. Impute missing value

* There are missing values
* There are outliers


Below are the steps involved to understand, clean and prepare your data for building your predictive model:

1. Variable Identification
2. Univariate Analysis
3. Bi-variate Analysis
4. Missing values treatment
5. Outlier treatment
6. Variable transformation
7. Variable creation



In [ ]:
# Check missing values in the dataset
dataset.apply(lambda x: sum(x.isnull()),axis=0) # axis 0 means column wise. 1 means row wise.

In [ ]:
dataset.shape

In [ ]:
# filling NA by mean
dataset['LoanAmount'].fillna(dataset['LoanAmount'].mean(), inplace=True)

In [ ]:
# imputing Mode 
dataset['Self_Employed'].value_counts()

#~86% values are NO. It is same to impute NO

dataset['Self_Employed'].fillna('No',inplace=True)

In [ ]:
# How to treat for extreme values in distribution of LoanAmount and ApplicantIncome?

# Let’s analyze LoanAmount first. 
# Since the extreme values are practically possible, 
# i.e. some people might apply for high value loans due to specific needs. 
# So instead of treating them as outliers, let’s try a log transformation to nullify their effect:


In [ ]:
dataset['LoanAmount'].hist(bins=20)

In [ ]:
dataset['LoanAmount_log'] = numpy.log(dataset['LoanAmount'])
dataset['LoanAmount_log'].hist(bins=20)

# Now the distribution looks much closer to normal and effect of extreme values has been significantly subsided.

In [ ]:
dataset['TotalIncome'] = dataset['ApplicantIncome'] + dataset['CoapplicantIncome']
dataset['TotalIncome'].hist(bins=20)

In [ ]:
dataset['TotalIncome_log'] = numpy.log(dataset['TotalIncome'])
dataset['TotalIncome_log'].hist(bins=20)

In [ ]:
dataset['Gender'].fillna(dataset['Gender'].mode()[0], inplace=True)
dataset['Married'].fillna(dataset['Married'].mode()[0], inplace=True)
dataset['Dependents'].fillna(dataset['Dependents'].mode()[0], inplace=True)
dataset['Loan_Amount_Term'].fillna(dataset['Loan_Amount_Term'].mode()[0], inplace=True)
dataset['Credit_History'].fillna(dataset['Credit_History'].mode()[0], inplace=True)

In [ ]:
# Check missing values in the dataset
dataset.apply(lambda x: sum(x.isnull()),axis=0) # axis 0 means column wise. 1 means row wise.

In [ ]:
display(dataset.head(10))

In [ ]:
# One hot encoding

from sklearn.preprocessing import LabelEncoder
var_mod = ['Gender','Married','Dependents','Education','Self_Employed','Property_Area','Loan_Status']
le = LabelEncoder()

for i in var_mod:
    dataset[i] = le.fit_transform(dataset[i])
dataset.dtypes 

# Build Models

Let’s evaluate 6 different algorithms:

1. Logistic Regression (LR)
2. Linear Discriminant Analysis (LDA)
3. K-Nearest Neighbors (KNN).
4. Classification and Regression Trees (CART).
5. Gaussian Naive Bayes (NB).
6. Support Vector Machines (SVM).

In [ ]:
dataset.head()

In [ ]:
# Split-out validation dataset
X = dataset[['Gender','Married','Dependents','Education','Self_Employed','LoanAmount_log','Loan_Amount_Term','Credit_History','Property_Area','TotalIncome_log']].values
Y = dataset['Loan_Status'].values

validation_size = 0.20
seed = 100
X_train, X_test, Y_train, Y_test = sklearn.model_selection.train_test_split(X, Y, test_size=validation_size, random_state=seed)

In [ ]:
# Algorithms
model_LR=sklearn.linear_model.LogisticRegression()
model_LDA=sklearn.discriminant_analysis.LinearDiscriminantAnalysis()
model_KNN= sklearn.neighbors.KNeighborsClassifier()
model_CART=sklearn.tree.DecisionTreeClassifier()
model_NB=sklearn.naive_bayes.GaussianNB()
model_SVM=sklearn.svm.SVC()

In [ ]:
# Fitting Model
model_LR.fit(X_train,Y_train)
model_LDA.fit(X_train,Y_train)
model_KNN.fit(X_train,Y_train)
model_CART.fit(X_train,Y_train)
model_NB.fit(X_train,Y_train)
model_SVM.fit(X_train,Y_train)

In [ ]:
# Making Predictions

# for Train dataset
trainResult_LR=model_LR.predict(X_train)
trainResult_LDA=model_LDA.predict(X_train)
trainResult_KNN=model_KNN.predict(X_train)
trainResult_CART=model_CART.predict(X_train)
trainResult_NB=model_NB.predict(X_train)
trainResult_SVM=model_SVM.predict(X_train)


# for test dataset
testResult_LR=model_LR.predict(X_test)
testResult_LDA=model_LDA.predict(X_test)
testResult_KNN=model_KNN.predict(X_test)
testResult_CART=model_CART.predict(X_test)
testResult_NB=model_NB.predict(X_test)
testResult_SVM=model_SVM.predict(X_test)


In [ ]:
#Combine result of all models
trainResult=pandas.DataFrame([Y_train,trainResult_LR,trainResult_LDA,trainResult_KNN,trainResult_CART,trainResult_NB,trainResult_SVM]).T
trainResult.columns=['Actual','LR','LDA','KNN','CART','NB','SVM']
display(trainResult.head(30))

### 3. Evaluation
1. Accuracy score
2. Confusion Matrix
3. Classification Report



#### Logistic Regression Evaluation

In [ ]:
# Train

print(sklearn.metrics.accuracy_score(Y_train, trainResult_LR))
print(sklearn.metrics.confusion_matrix(Y_train, trainResult_LR))
print(sklearn.metrics.classification_report(Y_train, trainResult_LR))
print('--------------------------------------------------------------')

# Test
print(sklearn.metrics.accuracy_score(Y_test, testResult_LR))
print(sklearn.metrics.confusion_matrix(Y_test, testResult_LR))
print(sklearn.metrics.classification_report(Y_test, testResult_LR))

#### Linear Discriminant Analysis (LDA) Evaluation

In [ ]:
# Train

print(sklearn.metrics.accuracy_score(Y_train, trainResult_LDA))
print(sklearn.metrics.confusion_matrix(Y_train, trainResult_LDA))
print(sklearn.metrics.classification_report(Y_train, trainResult_LDA))
print('--------------------------------------------------------------')

# Test
print(sklearn.metrics.accuracy_score(Y_test, testResult_LDA))
print(sklearn.metrics.confusion_matrix(Y_test, testResult_LDA))
print(sklearn.metrics.classification_report(Y_test, testResult_LDA))

#### K-Nearest Neighbors (KNN) Evaluation

In [ ]:
# Train

print(sklearn.metrics.accuracy_score(Y_train, trainResult_KNN))
print(sklearn.metrics.confusion_matrix(Y_train, trainResult_KNN))
print(sklearn.metrics.classification_report(Y_train, trainResult_KNN))
print('--------------------------------------------------------------')

# Test
print(sklearn.metrics.accuracy_score(Y_test, testResult_KNN))
print(sklearn.metrics.confusion_matrix(Y_test, testResult_KNN))
print(sklearn.metrics.classification_report(Y_test, testResult_KNN))

#### Classification and Regression Trees (CART) Evaluation

Decision tree has overfitted the training dataset. 100% accuracy in train dataset and low accuracy in test dataset.

In [ ]:
# Train

print(sklearn.metrics.accuracy_score(Y_train, trainResult_CART))
print(sklearn.metrics.confusion_matrix(Y_train, trainResult_CART))
print(sklearn.metrics.classification_report(Y_train, trainResult_CART))
print('--------------------------------------------------------------')

# Test
print(sklearn.metrics.accuracy_score(Y_test, testResult_CART))
print(sklearn.metrics.confusion_matrix(Y_test, testResult_CART))
print(sklearn.metrics.classification_report(Y_test, testResult_CART))



#### Gaussian Naive Bayes (NB) Evaluation

In [ ]:
# Train

print(sklearn.metrics.accuracy_score(Y_train, trainResult_NB))
print(sklearn.metrics.confusion_matrix(Y_train, trainResult_NB))
print(sklearn.metrics.classification_report(Y_train, trainResult_NB))
print('--------------------------------------------------------------')

# Test
print(sklearn.metrics.accuracy_score(Y_test, testResult_NB))
print(sklearn.metrics.confusion_matrix(Y_test, testResult_NB))
print(sklearn.metrics.classification_report(Y_test, testResult_NB))

#### Support Vector Machines (SVM) Evaluation

In [ ]:
# Train

print(sklearn.metrics.accuracy_score(Y_train, trainResult_SVM))
print(sklearn.metrics.confusion_matrix(Y_train, trainResult_SVM))
print(sklearn.metrics.classification_report(Y_train, trainResult_SVM))
print('--------------------------------------------------------------')

# Test
print(sklearn.metrics.accuracy_score(Y_test, testResult_SVM))
print(sklearn.metrics.confusion_matrix(Y_test, testResult_SVM))
print(sklearn.metrics.classification_report(Y_test, testResult_SVM))

## IF YOU LIKED THIS NOTEBOOK SO PLEASE DO UPVOTE.THANK YOU